# quick_start_hls4ml — Keras → hls4ml → dfx4ml (kv260)

End-to-end flow with the **VitisUnifiedDFx4ml** hls4ml backend plugin:

1. build a small skip-connection conv net and cut it into partitions
   (both a **2-partition** halfA/halfB split and a **4-partition** part1..part4 split);
2. convert each partition to a multi-port flat AXI-Stream HLS kernel for **kv260**;
3. (optional) **csim** each partition against Keras;
4. compute the **dfx_streamer** allocation and the `HwBuildHelper` params — *stop and inspect*;
5. **C synthesis** — *separate cell, natural pause*;
6. **FIFO-depth optimization** — *separate cell, natural pause*;
7. package the kernels (ip_catalog) + stitch the `create_dfx_region_user_bd` TCL;
8. run the dfx4ml **hardware build** and **software** export.

> The heavy stages (5–8) require Vitis / Vivado on PATH and take a long time.
> They are isolated in their own cells so the notebook stops between them.

In [1]:
# ── Cell 1: environment + plugin registration ───────────────────────────────
import os, sys, json, shutil
from pathlib import Path

REPO = Path.cwd()
sys.path.insert(0, str(REPO / 'lib'))            # lib/hls4ml_con + lib/dfx_streamer_cal
sys.path.insert(0, str(REPO / 'hls4ml'))         # hls4ml submodule source
os.environ['HLS4ML_BACKEND_PLUGINS'] = 'hls4ml_con'   # discovered at `import hls4ml`
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import hls4ml
from hls4ml_con import streamer_glue

assert 'vitisunifieddfx4ml' in hls4ml.backends.get_available_backends(), \
    'VitisUnifiedDFx4ml backend not registered — check HLS4ML_BACKEND_PLUGINS / sys.path'
print('backends:', hls4ml.backends.get_available_backends())
print('OK: VitisUnifiedDFx4ml registered')

backends: ['vivado', 'vivadoaccelerator', 'vitis', 'vitisunified', 'quartus', 'catapult', 'symbolicexpression', 'oneapi', 'libero', 'vitisunifieddfx4ml']
OK: VitisUnifiedDFx4ml registered


## 1. Model + partitions

A small 2-D conv encoder/decoder with skip connections and a classifier head
(input `8×8×1` → 4-way `dense_out`), mirroring
`simple_conv_nn_skip4_vitis_unified.py`. The encoder emits three inter-partition
tensors — `bneck (2,2,8)`, `skip2 (4,4,16)`, `skip1 (8,8,8)` — and the 4-part split
adds a fourth, the encoder→decoder main feed `p3 (4,4,16)`.

In [ ]:
# ── Cell 2: build the full model and its composable stages ──────────────────
# Architecture mirrors simple_conv_nn_skip4_vitis_unified.py (U-Net + classifier head).
def enc_a(x):                       # inp(8,8,1) -> (p1_main(4,4,16), skip1(8,8,8))
    s1 = layers.Conv2D(8, 3, padding='same', activation='relu', name='enc_conv1')(x)   # (8,8,8) -> skip1
    x = layers.Conv2D(16, 3, padding='same', activation='relu', name='enc_conv1b')(s1)  # (8,8,16)
    x = layers.Conv2D(16, 3, padding='same', activation='relu', name='enc_conv2')(x)    # (8,8,16)
    p1 = layers.MaxPool2D(2, name='enc_pool1')(x)                  # (4,4,16)
    return p1, s1

def enc_b(p1):                      # p1_main(4,4,16) -> (bneck(2,2,8), skip2(4,4,16))
    s2 = layers.Conv2D(16, 3, padding='same', activation='relu', name='enc_conv3')(p1)  # (4,4,16) -> skip2
    x = layers.MaxPool2D(2, name='enc_pool2')(s2)                  # (2,2,16)
    bn = layers.Conv2D(8, 3, padding='same', activation='relu', name='bottleneck')(x)   # (2,2,8)
    return bn, s2

def dec_a(bn, s2):                  # (bneck, skip2) -> p3(4,4,16)
    y = layers.UpSampling2D(2, name='dec_up1')(bn)                 # (4,4,8)
    y = layers.Conv2D(16, 3, padding='same', activation='relu', name='dec_conv1')(y)    # (4,4,16)
    y = layers.Add(name='skip2_add')([y, s2])                      # fuse skip2 (4,4,16)
    return layers.Conv2D(16, 3, padding='same', activation='relu', name='dec_conv1b')(y)  # (4,4,16)

def dec_b(p3, s1):                  # (p3(4,4,16), skip1(8,8,8)) -> out(4,)
    y = layers.UpSampling2D(2, name='dec_up2')(p3)                 # (8,8,16)
    y = layers.Conv2D(8, 3, padding='same', activation='relu', name='dec_conv2')(y)     # (8,8,8)
    y = layers.Add(name='skip1_add')([y, s1])                      # fuse skip1 (8,8,8)
    y = layers.GlobalAveragePooling2D(name='gap')(y)
    y = layers.Dense(64, activation='relu', name='dense1')(y)
    return layers.Dense(4, activation=None, name='dense_out')(y)   # (4,)

inp = keras.Input(shape=(8, 8, 1), name='in0')
_p1, _s1 = enc_a(inp)
_bn, _s2 = enc_b(_p1)
_p3 = dec_a(_bn, _s2)
_out = dec_b(_p3, _s1)
full_model = keras.Model(inp, _out, name='full')
full_model.summary(line_length=90)

In [ ]:
# ── Cell 3: derive the partition sub-models ─────────────────────────────────
# 2-partition: halfA (encoder, 1 in / 3 out) -> halfB (decoder, 3 in / 1 out)
hA_in = keras.Input((8, 8, 1), name='hA_in')
hA_p1, hA_s1 = enc_a(hA_in); hA_bn, hA_s2 = enc_b(hA_p1)
halfA = keras.Model(hA_in, [hA_bn, hA_s2, hA_s1], name='halfA')   # out order: bneck, skip2, skip1

hB_bn = keras.Input((2, 2, 8), name='hB_bneck')
hB_s2 = keras.Input((4, 4, 16), name='hB_skip2')
hB_s1 = keras.Input((8, 8, 8), name='hB_skip1')
halfB = keras.Model([hB_bn, hB_s2, hB_s1], dec_b(dec_a(hB_bn, hB_s2), hB_s1), name='halfB')

# 4-partition: part1 (enc-A) -> part2 (enc-B) -> part3 (dec-A) -> part4 (dec-B)
p1_in = keras.Input((8, 8, 1), name='p1_in'); p1_main, p1_s1 = enc_a(p1_in)
part1 = keras.Model(p1_in, [p1_main, p1_s1], name='part1')        # main(4,4,16) -> part2 ; skip1(8,8,8) -> part4
p2_in = keras.Input((4, 4, 16), name='p2_in')
part2 = keras.Model(p2_in, list(enc_b(p2_in)), name='part2')      # -> bneck, skip2
p3_bn = keras.Input((2, 2, 8), name='p3_bneck'); p3_s2 = keras.Input((4, 4, 16), name='p3_skip2')
part3 = keras.Model([p3_bn, p3_s2], dec_a(p3_bn, p3_s2), name='part3')   # -> p3
p4_p3 = keras.Input((4, 4, 16), name='p4_p3'); p4_s1 = keras.Input((8, 8, 8), name='p4_skip1')
part4 = keras.Model([p4_p3, p4_s1], dec_b(p4_p3, p4_s1), name='part4')   # -> out

print('halfA out:', [o.shape for o in halfA.outputs])
print('part models built:', [m.name for m in (part1, part2, part3, part4)])

## 2. Conversion config

`backend='VitisUnifiedDFx4ml'`, `board='kv260'`, `axi_mode='axi_stream'`,
`package_as_xo=False` (ip_catalog). `input_flat` / `output_flat` mark which sides
connect to dfx streamers (flat, multi-port) vs the DMA (non-flat).

In [ ]:
# ── Cell 4: conversion helpers ──────────────────────────────────────────────
PREC = 'ap_fixed<16,6>'
PREC_BITS = 16
REUSE = 8           # matches HLS_REUSE_FACTOR in simple_conv_nn_skip4_vitis_unified.py
HLS_PARAMS = dict(
    backend='VitisUnifiedDFx4ml', io_type='io_stream', board='kv260',
    part='xck26-sfvc784-2LV-c', clock_period='10ns',
    input_type='float', output_type='float', axi_mode='axi_stream',
)
OUT_ROOT = REPO / 'hls4ml_dfx_out'

def _cfg(model):
    c = hls4ml.utils.config_from_keras_model(model, granularity='name')
    c['Model']['Strategy'] = 'Resource'
    c['Model']['ReuseFactor'] = REUSE
    c['Model']['Precision'] = PREC
    return c

def convert(spec, fifo_depths=None):
    '''spec: dict with name, model, input_flat, output_flat (+ topology fields).'''
    model = spec['model']
    cfg = _cfg(model)
    if fifo_depths is not None:
        cfg['Flows'] = ['vitisunifieddfx4ml:fifo_depth_optimization']
    hm = hls4ml.converters.convert_from_keras_model(
        model, hls_config=cfg, output_dir=str(OUT_ROOT / spec['name']),
        project_name=spec['project'], input_flat=spec['input_flat'],
        output_flat=spec['output_flat'], package_as_xo=False, **HLS_PARAMS)
    hm.write()
    return hm

In [ ]:
# ── Cell 5: choose a split and declare its partitions + inter-partition streams
# Flip SPLIT to '2part' or '4part'.
SPLIT = '2part'

def _shape(t):   # keras tensor shape without batch
    return tuple(int(d) for d in t.shape[1:])

CONFIGS = {
 '2part': dict(
    amt_phase=1, num_regions=2,
    specs=[
      dict(name='halfA', project='p_halfA', model=halfA, region=0, rm=0,
           input_flat=False, output_flat=True,
           inputs=['DMA'], outputs=['bneck', 'skip2', 'skip1']),
      dict(name='halfB', project='p_halfB', model=halfB, region=1, rm=0,
           input_flat=True, output_flat=False,
           inputs=['bneck', 'skip2', 'skip1'], outputs=['DMA']),
    ],
    streams=[
      dict(name='bneck', shape=_shape(hA_bn), precision=PREC_BITS, region=0, alloc_phase=0, free_phase=1),
      dict(name='skip2', shape=_shape(hA_s2), precision=PREC_BITS, region=0, alloc_phase=0, free_phase=1),
      dict(name='skip1', shape=_shape(hA_s1), precision=PREC_BITS, region=1, alloc_phase=0, free_phase=1),
    ]),
 '4part': dict(
    amt_phase=3, num_regions=2,
    specs=[
      dict(name='part1', project='p_part1', model=part1, region=0, rm=0,
           input_flat=False, output_flat=True, inputs=['DMA'], outputs=['a1', 'skip1']),
      dict(name='part2', project='p_part2', model=part2, region=1, rm=0,
           input_flat=True, output_flat=True, inputs=['a1'], outputs=['bneck', 'skip2']),
      dict(name='part3', project='p_part3', model=part3, region=0, rm=1,
           input_flat=True, output_flat=True, inputs=['bneck', 'skip2'], outputs=['p3']),
      dict(name='part4', project='p_part4', model=part4, region=1, rm=1,
           input_flat=True, output_flat=False, inputs=['p3', 'skip1'], outputs=['DMA']),
    ],
    streams=[
      dict(name='a1',    shape=_shape(p1_main), precision=PREC_BITS, region=0, alloc_phase=0, free_phase=1),
      dict(name='skip1', shape=_shape(p1_s1),   precision=PREC_BITS, region=1, alloc_phase=0, free_phase=2),
      dict(name='bneck', shape=_shape(hA_bn),   precision=PREC_BITS, region=1, alloc_phase=1, free_phase=2),
      dict(name='skip2', shape=_shape(hA_s2),   precision=PREC_BITS, region=1, alloc_phase=1, free_phase=2),
      dict(name='p3',    shape=_shape(_p3),     precision=PREC_BITS, region=0, alloc_phase=2, free_phase=3),
    ]),
}
CFG = CONFIGS[SPLIT]
print('SPLIT =', SPLIT, '|', [s['name'] for s in CFG['specs']])

In [6]:
# ── Cell 6: convert every partition (writes firmware + dfx TCL fragment) ─────
hls_models = {}
for spec in CFG['specs']:
    print('=' * 50, 'convert', spec['name'])
    hls_models[spec['name']] = convert(spec)
print('converted:', list(hls_models))

================================================== convert halfA


================================================== convert halfB


converted: ['halfA', 'halfB']


## 3. (optional) csim each partition vs Keras

Compiles the C++ bridge and runs `predict`. Multi-output partitions return a flat
concatenation in output-port order. Skip if you only want the hardware build.

In [ ]:
# ── Cell 7: csim sanity check (optional, needs a C++ toolchain) ─────────────
RUN_CSIM = True
if RUN_CSIM:
    x = np.random.rand(2, 8, 8, 1).astype('float32')
    for spec in CFG['specs']:
        hm = hls_models[spec['name']]
        try:
            hm.compile()
            print(spec['name'], 'csim ok; output shape', np.asarray(hm.predict(x)).shape)
        except Exception as e:
            print(spec['name'], 'csim skipped:', type(e).__name__, e)
else:
    print('RUN_CSIM=False — skipping csim')

## 4. dfx_streamer allocation  →  HwBuildHelper params  *(inspect & pause)*

Runs **before** any long synthesis. Review the streamer table and the three
`HwBuildHelper` lists; adjust precision / `total_banks` if a bus width is not a
power of two, then continue to synthesis.

In [8]:
# ── Cell 8: compute dfx params + stitch the user-BD dispatcher TCL ──────────
TOTAL_BANKS = 64
USER_RM_TCL = str(OUT_ROOT / 'create_dfx_region_user_bd.tcl')

dfx = streamer_glue.compute_dfx_params(
    partitions=CFG['specs'], streams=CFG['streams'],
    total_banks=TOTAL_BANKS, amt_phase=CFG['amt_phase'], num_regions=CFG['num_regions'])

print('\n=== dfx_streamers (index 0 = DMA) ===')
for i, s in enumerate(dfx['dfx_streamers']):
    print(' ', i, s)
print('=== dfx_regions ===')
for r, d in enumerate(dfx['dfx_regions']):
    print(' ', r, d)
print('=== rm_schemetics ===')
for r, region in enumerate(dfx['rm_schemetics']):
    for m, rm in enumerate(region):
        print(f'  region {r} rm {m}: {rm}')

streamer_glue.build_dispatcher_tcl(CFG['specs'], USER_RM_TCL)
print('\nuser_rm_build_tcl ->', USER_RM_TCL)

[dfx-streamer] streams:
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  | name       | shape              | region | alloc_phase | free_phase | precision | amt_entry_per_query | bits_per_entry | amt_banks_per_entry | amt_query_per_bankGrp |
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  | bneck      | (2, 2, 8)          |      0 |           0 |          1 |        16 |                   4 |            128 |                   2 |                  1024 |
  | skip2      | (4, 4, 8)          |      0 |           0 |          1 |        16 |                  16 |            128 |                   2 |                   256 |
  | skip1      | (8, 8, 4)          |      1 |           0 |          1 |        16 |                  64 |             64 |                   1 |               

## 5. C synthesis  *(separate cell — long-running, pauses here)*

Synthesizes + packages each partition as an ip_catalog IP. Requires `v++` /
`vitis-run` on PATH.

In [9]:
# ── Cell 9: synthesize each partition ───────────────────────────────────────
RUN_SYNTH = False
if RUN_SYNTH:
    for spec in CFG['specs']:
        print('=' * 50, 'synth', spec['name'])
        hls_models[spec['name']].build(synth=True, csim=False, cosim=False, log_to_stdout=True)
    print('synthesis done')
else:
    print('RUN_SYNTH=False — set True to synthesize (needs Vitis on PATH)')

RUN_SYNTH=False — set True to synthesize (needs Vitis on PATH)


## 6. FIFO-depth optimization  *(separate cell — pauses here)*

Runs the `vitisunifieddfx4ml:fifo_depth_optimization` flow on the **full** model,
producing `fifo_depths.json`; the optimized depths are then applied to the
partition sub-models (re-convert with `fifo_depths`).

In [10]:
# ── Cell 10: fifo-depth optimization on the full model ──────────────────────
RUN_FIFO = False
FULL_DIR = OUT_ROOT / 'full'
if RUN_FIFO:
    full_cfg = _cfg(full_model)
    full_cfg['Flows'] = ['vitisunifieddfx4ml:fifo_depth_optimization']
    full_hls = hls4ml.converters.convert_from_keras_model(
        full_model, hls_config=full_cfg, output_dir=str(FULL_DIR),
        project_name='p_full', input_flat=False, output_flat=False,
        package_as_xo=False, **HLS_PARAMS)
    full_hls.build(synth=True, fifo_opt=True, log_to_stdout=False)
    fj = FULL_DIR / 'fifo_depths.json'
    fifo_depths = json.load(open(fj)) if fj.exists() else None
    print('fifo depths:', None if fifo_depths is None else len(fifo_depths), 'entries')
    # re-convert partitions inheriting the optimized depths
    for spec in CFG['specs']:
        hls_models[spec['name']] = convert(spec, fifo_depths=fifo_depths)
    print('partitions re-converted with optimized FIFO depths')
else:
    print('RUN_FIFO=False — set True to run FIFO optimization (needs Vitis)')

RUN_FIFO=False — set True to run FIFO optimization (needs Vitis)


## 7–8. Package + dfx4ml hardware build, then software export

In [ ]:
# ── Cell 11: dfx4ml hardware build ──────────────────────────────────────────
from lib.hw_build import HwBuildHelper
from lib.sw_build import SwBuildHelper

RUN_HWBUILD = False
VIVADO_PATH = '/tools/Xilinx/Vivado/2023.2/bin/vivado'

if RUN_HWBUILD:
    hw_builder = HwBuildHelper(
        build_folder_path='./build_prj',
        dfx_root_path='.',
        board='kv260',
        user_repo_path=str(OUT_ROOT),               # consolidated packaged-IP location
        user_rm_build_tcl_path=USER_RM_TCL,         # generated create_dfx_region_user_bd
        req_gen_ip=1,
        num_core=4,
        clk_frq=99999001,
        rm_index_width=3,
        dfx=dfx,                                    # streamers/regions/rm_schemetics unpacked in HwBuildHelper
        test_mode=0,                                # user kernels
        vivado_path=VIVADO_PATH,
        export_folder_path='./export',
    )
    hw_builder.run_build()
    hw_builder.package_export_files()
    print('hardware build complete')
else:
    print('RUN_HWBUILD=False — set True to run the dfx4ml Vivado build')

In [12]:
# ── Cell 12: software / PYNQ export ─────────────────────────────────────────
if RUN_HWBUILD:
    sw_builder = SwBuildHelper(hw_builder=hw_builder)
    sw_builder.package_export_file()
    print('software export complete -> ./export')
else:
    print('run the hardware build first (RUN_HWBUILD=True)')

run the hardware build first (RUN_HWBUILD=True)
